# 02 — Demo end-to-end trên **Google Colab + Google Drive**

Notebook này train mô hình Transformer trên Colab GPU, dữ liệu để trên Google Drive.
Mở bằng *extension Colab trong VS Code* hoặc trên colab.research.google.com.

**Luồng:** mount Drive → clone repo → (crawl hoặc dùng data có sẵn) → chỉ báo → train →
đánh giá → chọn Top-N → đóng gói `models/` + `final_merged.csv` để tải về chạy website.

## 0. Mount Drive, clone repo, cài đặt

In [1]:
from google.colab import drive
# force_remount=True để Colab nhận shortcut mới thêm vào My Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
# ⚠️ Đổi URL cho đúng repo của bạn nếu khác
REPO_URL = "https://github.com/HoangTechCS-AIE/MachineLearning-HaUI.git"
%cd /content
if not os.path.exists("/content/MachineLearning-HaUI"):
    !git clone $REPO_URL
%cd /content/MachineLearning-HaUI
!git pull origin main          # luôn cập nhật bản mới nhất
!pip install -q -r requirements.txt

/content
Cloning into 'MachineLearning-HaUI'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 104 (delta 9), reused 104 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 132.21 KiB | 6.30 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/MachineLearning-HaUI
From https://github.com/HoangTechCS-AIE/MachineLearning-HaUI
 * branch            main       -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.8 MB/s eta 0:00:00


In [3]:
import sys; sys.path.insert(0, '.')
from pathlib import Path
from src.config import load_config

# Dùng cấu hình Colab (paths trỏ Google Drive). Sửa BASE trong file này nếu cần:
#   config/config.colab.yaml
cfg = load_config("config/config.colab.yaml")
for p in [cfg.paths.raw_dir, cfg.paths.processed_dir, cfg.paths.model_dir, cfg.paths.figures_dir]:
    Path(cfg.abs_path(p)).mkdir(parents=True, exist_ok=True)
print("raw  :", cfg.abs_path(cfg.paths.raw_dir))
print("model:", cfg.abs_path(cfg.paths.model_dir))
print("merged:", cfg.abs_path(cfg.paths.merged_file))

raw  : /content/drive/MyDrive/btl-hocmay/data/raw
model: /content/drive/MyDrive/btl-hocmay/models
merged: /content/drive/MyDrive/btl-hocmay/data/processed/final_merged.csv


## 1. Chuẩn bị dữ liệu (chọn 1 trong 2)

- **Cách A — crawl ngay trên Colab** (cần tài khoản FiinQuant): bỏ comment ô dưới.
- **Cách B — đã đẩy data lên Drive**: bỏ qua ô crawl, chỉ cần có `final_merged.csv`
  (hoặc các CSV thô trong `raw_dir`) trên Drive.

In [4]:
# === Cách A: crawl trên Colab (cần tài khoản FiinQuant) ===
# import os
# os.environ["FIINQUANT_USERNAME"] = "TEN_DANG_NHAP"
# os.environ["FIINQUANT_PASSWORD"] = "MAT_KHAU"
# !pip install -q --extra-index-url https://fiinquant.github.io/fiinquantx/simple fiinquantx
# from src.data.crawl import crawl_all
# crawl_all(cfg)

### Cách C — dùng **dataset DSCT có sẵn trên Drive** (không cần FiinQuant)

Schema DSCT khớp pipeline này (cùng tên cột `timestamp/ticker/OHLCV` và chỉ báo).
**Yêu cầu:** folder `dataset` phải nằm trong *My Drive* — nếu đang ở *Shared with me*,
chuột phải folder → **Add shortcut to Drive** → My Drive, rồi chạy lại cell mount ở Bước 0.

Ô dưới **tự dò** folder dữ liệu (ưu tiên `Filler_Data` > `Data_Preprocess` > `Raw_Data`),
kể cả khi CSV nằm trong thư mục con. `build_features` sẽ tính lại chỉ báo (kèm `vma20`).

In [5]:
import glob, os, pandas as pd

def find_dsct_dir():
    """Dò folder chứa CSV của DSCT trong My Drive."""
    for name in ["Filler_Data", "Data_Preprocess", "Raw_Data"]:
        for folder in sorted(glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)):
            csvs = glob.glob(f"{folder}/**/*.csv", recursive=True)
            if csvs:
                return folder, sorted(csvs)
    return None, []

DSCT_DIR, files = find_dsct_dir()
assert DSCT_DIR, (
    "Chưa thấy CSV của DSCT trong My Drive.\n"
    "→ Add shortcut folder 'dataset' vào My Drive, RỒI CHẠY LẠI cell mount ở Bước 0 "
    "(Colab cần remount mới thấy shortcut mới)."
)
cfg.paths.raw_dir = DSCT_DIR
print("DSCT_DIR =", DSCT_DIR)
print(f"{len(files)} file:", [os.path.basename(f) for f in files][:6])
_sample = pd.read_csv(files[0], nrows=3)
print("Cột:", list(_sample.columns))
_sample

DSCT_DIR = /content/drive/MyDrive/dataset/Data_Preprocess
1 file: ['final_merged.csv']
Cột: ['timestamp', 'ticker', 'score', 'open', 'high', 'low', 'close', 'volume']


,timestamp,ticker,score,open,high,low,close,volume
0,2025-01-02,AAV,1.0,7300.0,7500.0,7300.0,7400.0,182600.0
1,2025-01-03,AAV,-0.5,7400.0,7400.0,7200.0,7200.0,762800.0
2,2025-01-06,AAV,-0.5,7200.0,7400.0,7100.0,7200.0,446300.0


## 2. Tính chỉ báo kỹ thuật + gộp dữ liệu
Tạo `final_merged.csv` trên Drive (bỏ qua nếu đã có).

In [ ]:
from src.data.indicators import build_features
# Luôn build lại từ data DSCT vừa trỏ ở Cách C (tránh dùng final_merged cũ/nhỏ trên Drive)
build_features(cfg)

## 3. Train trên GPU

In [7]:
import torch
print("CUDA khả dụng:", torch.cuda.is_available())
from src.train import train
out = train(cfg)
print("best val L1 =", out["best_val"])

CUDA khả dụng: True
[train] device = cuda
[train] train samples = 133811 | val samples = 0
[train] epoch   1/150 | train L1 0.08967 | val L1 nan | 21.9s
[train] epoch   2/150 | train L1 0.06712 | val L1 nan | 20.0s
[train] epoch   3/150 | train L1 0.06291 | val L1 nan | 20.9s
[train] epoch   4/150 | train L1 0.05998 | val L1 nan | 20.2s
[train] epoch   5/150 | train L1 0.05839 | val L1 nan | 20.5s
[train] epoch   6/150 | train L1 0.05692 | val L1 nan | 20.4s
[train] epoch   7/150 | train L1 0.05613 | val L1 nan | 20.5s
[train] epoch   8/150 | train L1 0.05541 | val L1 nan | 20.7s
[train] epoch   9/150 | train L1 0.05478 | val L1 nan | 20.4s
[train] epoch  10/150 | train L1 0.05419 | val L1 nan | 21.0s
[train] epoch  11/150 | train L1 0.05369 | val L1 nan | 19.9s
[train] epoch  12/150 | train L1 0.05300 | val L1 nan | 20.8s
[train] epoch  13/150 | train L1 0.05249 | val L1 nan | 20.1s
[train] epoch  14/150 | train L1 0.05218 | val L1 nan | 20.8s
[train] epoch  15/150 | train L1 0.05204 

## 4. Đánh giá trên tập test

In [8]:
from src.evaluate import evaluate
res = evaluate(cfg, plot=True)
print("Transformer:", res["metrics"])
print("Naive      :", res["naive"])

RuntimeError: Tập test rỗng.

## 5. Chọn Top-N cổ phiếu

In [9]:
from src.predict import predict_test, select_top_stocks
pred = predict_test(cfg)
months = sorted(pred["timestamp"].dt.strftime("%Y-%m").unique())
print("Các tháng test:", months)
select_top_stocks(pred, months[-1], cfg.select.top_n)

AttributeError: Can only use .dt accessor with datetimelike values

## 6. Đóng gói artifacts để tải về chạy **website**

Tải `btl_artifacts.zip` về máy, giải nén rồi đặt:
- thư mục `models/*` → `models/` của repo local
- `final_merged.csv` → `data/processed/` của repo local

Sau đó chạy backend (`uvicorn backend.app:app`) là website dùng được model thật.

In [10]:
import shutil, os
os.makedirs("/content/export/data/processed", exist_ok=True)
shutil.copytree(cfg.abs_path(cfg.paths.model_dir), "/content/export/models", dirs_exist_ok=True)
shutil.copy(cfg.abs_path(cfg.paths.merged_file), "/content/export/data/processed/")
shutil.make_archive("/content/btl_artifacts", "zip", "/content/export")
from google.colab import files
files.download("/content/btl_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>